# 10_tree_queue_pipeline

本章通过一个可运行的 Ascend C 教学实验，把树的层次依赖、优先队列和 Ascend C 的三阶段流水线放到同一个任务调度问题中。实验先用 BFS 队列推进依赖层级，再用小根堆维护已就绪任务，最后比较 FIFO 顺序和优先级顺序在 `CopyIn → Compute → CopyOut` 流水线中的完成时间。

## 0. 前置要求与环境

开始本章前，请先掌握数组、队列、树和堆的基本概念，能够阅读 Python 函数和简单的 C/C++ 控制流。Python reference 只需要 Python 3.8+ 标准库；如果要执行 910B 设备部分，还需要 Linux、已验证的 CANN 9.1.0、`bash`、`cmake`、`gcc` 和 Ascend 910B3 环境。没有设备时，可以先完成本章的概念学习和 Python reference。

本章共有三个小节：10.01 介绍树形依赖与流水线，10.02 完成 Python reference 和 910B 工程操作，10.03 通过选择、填空和编程实践检查学习结果。

## 1. 实验背景

调度问题可以表示为一棵树：父任务完成后，子任务才进入就绪队列。BFS 用 FIFO 队列逐层释放任务，优先队列用堆从已就绪任务中选择关键路径更长的任务，流水线再用双缓冲安排搬运、计算和写回。

本实验不重复 Reduce、TopK 或 MoE 路由主题，重点是“树形依赖如何转化为数组索引、就绪任务如何进入队列、流水线如何通过双缓冲隐藏阶段等待”。

## 2. 树形依赖与队列数据结构

树的连接关系用 `parent[i] = p` 表示：节点 `i` 的父节点是 `p`，根节点没有父节点，因此使用 `-1`。这样可以把树的父子约束压缩成整数索引，避免指针跳转和递归。

- **BFS frontier**：从根节点出发，用 FIFO 队列逐层收集同一深度的节点，得到 `depth` 和层级列表。
- **FIFO 队列**：保持 frontier 的层序，先入队的任务先调度。
- **优先队列（小根堆）**：从已就绪任务中选择剩余子树工作量更大的任务优先执行。

## 3. FIFO 与优先队列调度

- **FIFO 调度**：按 BFS 层序依次执行任务，只保证依赖顺序，不区分不同分支后续还有多少工作。
- **优先队列调度**：从叶子向根累计 `subtree_work`，每次弹出一个已就绪任务，再把它的子节点放入堆中；堆键使用剩余子树工作量，优先推进更可能影响总时长的分支。

优先队列只能改变同一时刻已就绪任务的选择顺序，不能违反父子依赖——子节点必须等父节点完成后才会入队。

## 4. Ascend C 执行模型与 910B 适配

每个任务依次经历 `CopyIn → Compute → CopyOut` 三个阶段。`queue_depth=2` 表示两个可复用的缓冲槽，一个槽执行 `Compute`/`CopyOut` 时另一个槽可以开始下一次 `CopyIn`，从而隐藏阶段等待。

调度循环存在跨任务依赖，不能伪装成无依赖的逐元素并行算子。910B 上 `TreeQueuePipelineLite` 使用一个 control block 保持时序确定；Host 侧通过 tiling 数据传入 `taskCount`、`queueDepth`、`computeLanes`，Kernel 不写死树规模和层数，因此算法代码不依赖具体芯片型号。

## 5. 调度拓扑

<div style="text-align: left;">
  <img src="./images/tree_queue_pipeline.svg" alt="树形任务队列与双缓冲流水线示意图" width="720">
</div>

图中 FIFO 与优先队列是可选的就绪任务顺序；它们共享后续的 `CopyIn → Compute → CopyOut` 双缓冲流水线路径。

## 6. 学习目标与章节内容

完成本章后，你将能够：

1. 用 `parent[i]` 数组表示树的父子关系，并用 BFS frontier 计算层级和依赖释放顺序。
2. 用二叉堆实现就绪任务优先队列，比较 FIFO 与优先级调度的任务顺序。
3. 阅读 Ascend C Kernel 与 Host tiling 代码，理解 control block 与 tiling 参数的适配方式。
4. 使用 910B 目标构建自定义算子，并区分 Python reference、设备 `stage_end` 与端到端时间。

下一步：进入 [10.02 动手实验](./10.02_tree_queue_lab.ipynb)，完成工程检查、构建和数据分析；最后使用 [10.03 章节测试](./10.03_chapter_test.ipynb) 检查理解。

## 7. 课后练习

1. 解释为什么 `parent[i] = p` 可以表达树的父子依赖，以及根节点为什么使用 `-1`。
2. 说明 FIFO 和优先队列分别保持了什么约束，优先队列为什么不能让子节点绕过父节点提前执行。
3. 说明 `queue_depth=2` 在 `CopyIn → Compute → CopyOut` 中提供了什么资源条件。

请先独立作答，再运行下一个代码单元查看参考答案。

In [ ]:
!cat answer/10.01_chapter_intro.md